<a href="https://colab.research.google.com/github/xd2285-cloud/Data-Bootcamp-midterm-project/blob/main/Midterm_Project%E6%9C%80%E7%BB%88%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Project Introduction**

**Project** **Goal**

**Identifying** **Promising** **Restaurant** **Segments** **in** **Manhattan**:

**An** **Exploratory** **Data** **Analysis** **of** **Consumer** **Appeal** **and** **Operational** **Stability**

In this project, “promising” does not mean proven profitability.

It refers to stronger consumer appeal on Yelp and/or more stable operational performance in inspection data.

## **Step 0 Import Necessary Tools**

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import time

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

## **Step 1 Import API**

### **Yelp**

In [ ]:
Yelp_API_KEY = "kBJBw7w8Gkg1JR_nu5qfeAkony8eKXGLs8sA60RDtMG5YsBx4W5wSuOWSlmeg5FcjttMbaq9_kKd72cpOA-OcBrgSOwAVcfUy28T3LjBmrJ1um9VY3NHZDIEQYqwaXYx"

yelp_headers = {"Authorization": "Bearer " + Yelp_API_KEY}

In [ ]:
# Yelp search function
# term - search what
# limit - take how many pieces of information each time
# offset - start from which piece

def search_yelp_restaurants(location, term="restaurants", limit=50, offset=0):
    url = "https://api.yelp.com/v3/businesses/search"
    params = {
        "term": term,
        "location": location,
        "limit": limit,
        "offset": offset,
        "categories": "restaurants"}

    response = requests.get(url, headers=yelp_headers, params=params)
    return response.json()

In [ ]:
# Yelp search terms
# We use multiple search terms to expand coverage
# write-up note: set multiples ways to searching for restaurants because if we only set 'restaurant' as the terms
# the API only returns popular restaurants, leading to some bias
# Even though we manage to lower this subjective bias, the limitation of Yelp searching system still cannot provide us with fully thorough types of restaurants in Manhattan


terms = [
    "restaurants",
    "italian restaurants",
    "chinese restaurants",
    "japanese restaurants",
    "korean restaurants",
    "mexican restaurants",
    "thai restaurants",
    "indian restaurants",
    "pizza",
    "sushi",
    "coffee",
    "brunch",
    "bars",
    "bakeries"]

In [ ]:
# Pull Yelp data
# Yelp search API can only return limited results for each term
# so we use multiple terms and multiple offsets
# apply time.sleep() to avoid too frequent/fast request which may lead to error reports

all_businesses = []

for term in terms:
    print("Now searching term:", term)
    for offset in [0, 50, 100, 150]:
      data = search_yelp_restaurants("Manhattan, NY", term=term, limit=50, offset=offset)

      if isinstance(data, dict) and "businesses" in data:
        all_businesses.extend(data["businesses"])
        print("added:", len(data["businesses"]), "| current total:", len(all_businesses))
      else:
        print("Error:", data)

      time.sleep(1)

    data = search_yelp_restaurants("Manhattan, NY", term=term, limit=40, offset=200)
    if isinstance(data, dict) and "businesses" in data:
      all_businesses.extend(data["businesses"])
      print("added last page:", len(data["businesses"]), "| current total:", len(all_businesses))
    else:
      print("Error:", data)

    time.sleep(1)

Now searching term: restaurants
added: 50 | current total: 50
added: 50 | current total: 100
added: 50 | current total: 150
added: 50 | current total: 200
added last page: 40 | current total: 240
Now searching term: italian restaurants
added: 50 | current total: 290
added: 50 | current total: 340
added: 50 | current total: 390
added: 50 | current total: 440
added last page: 40 | current total: 480
Now searching term: chinese restaurants
added: 50 | current total: 530
added: 50 | current total: 580
added: 50 | current total: 630
added: 50 | current total: 680
added last page: 40 | current total: 720
Now searching term: japanese restaurants
added: 50 | current total: 770
added: 50 | current total: 820
added: 50 | current total: 870
added: 50 | current total: 920
added last page: 40 | current total: 960
Now searching term: korean restaurants
added: 50 | current total: 1010
added: 50 | current total: 1060
added: 50 | current total: 1110
added: 50 | current total: 1160
added last page: 40 |

In [ ]:
# Convert Yelp data to dataframe

yelp_df = pd.DataFrame(all_businesses)
print("raw yelp shape:", yelp_df.shape)
yelp_df.head()

In [ ]:
# as we use different search terms, some restaurants may be captured more than once
# so it's necessary to remove duplicates by the restaurants' unique id

yelp_df = yelp_df.drop_duplicates(subset="id")
print("after removing duplicates:", yelp_df.shape)

In [ ]:
# Quick overview of raw Yelp data

yelp_df.info()

In [ ]:
# Show Yelp raw columns

print("Yelp raw columns:")
print(list(yelp_df.columns))

### **NYC Open Data**

In [ ]:
# NYC Open Data API
# We directly filter Manhattan in the API URL

inspection_url = (
  "https://data.cityofnewyork.us/resource/43nn-pn8j.csv?"
  "$select=camis,dba,boro,zipcode,cuisine_description,inspection_date,"
  "action,violation_code,violation_description,critical_flag,score,grade"
  "&$where=boro='Manhattan'"
  "&$limit=50000")

inspection_df = pd.read_csv(inspection_url)

print(inspection_df.shape)
inspection_df.head()

In [ ]:
# Check borough values

inspection_df["boro"].value_counts(dropna=False)

In [ ]:
# Keep selected columns

inspection_clean = inspection_df.copy()

print(inspection_clean.shape)
inspection_clean.head()

In [ ]:
# Basic type cleaning for NYC inspection data
# score from str to values
# inspection_date from str to datetime

inspection_clean["score"] = pd.to_numeric(inspection_clean["score"], errors="coerce")
inspection_clean["inspection_date"] = pd.to_datetime(inspection_clean["inspection_date"], errors="coerce")

print(inspection_clean.shape)
inspection_clean.head()

In [ ]:
# Show NYC inspection columns

print("NYC inspection columns:")
print(list(inspection_clean.columns))

### **Dataset overview**

In [ ]:
print("Yelp shape:", yelp_df.shape)
print("NYC shape:", inspection_clean.shape)

print("\nYelp data types:")
print(yelp_df.dtypes)

print("\nNYC data types:")
print(inspection_clean.dtypes)

## **Step 2 Data Cleaning**

### **Yelp Cleaning**

In [ ]:
# rating - our defined term of success
# review_count - popularity
# price - market segmentation

yelp_clean = pd.DataFrame()

yelp_clean["id"] = yelp_df["id"]
yelp_clean["name"] = yelp_df["name"]
yelp_clean["rating"] = yelp_df["rating"]
yelp_clean["review_count"] = yelp_df["review_count"]
yelp_clean["is_closed"] = yelp_df["is_closed"]
yelp_clean["price"] = yelp_df["price"]
yelp_clean["phone"] = yelp_df["phone"]
yelp_clean["distance"] = yelp_df["distance"]

**data extraction**

In [ ]:
# Extract coordinates

yelp_clean["latitude"] = yelp_df["coordinates"].apply(lambda x: x.get("latitude") if isinstance(x, dict) else np.nan)
yelp_clean["longitude"] = yelp_df["coordinates"].apply(lambda x: x.get("longitude") if isinstance(x, dict) else np.nan)

In [ ]:
# Extract location information

yelp_clean["address1"] = yelp_df["location"].apply(lambda x: x.get("address1") if isinstance(x, dict) else np.nan)
yelp_clean["city"] = yelp_df["location"].apply(lambda x: x.get("city") if isinstance(x, dict) else np.nan)
yelp_clean["state"] = yelp_df["location"].apply(lambda x: x.get("state") if isinstance(x, dict) else np.nan)
yelp_clean["zip_code"] = yelp_df["location"].apply(lambda x: x.get("zip_code") if isinstance(x, dict) else np.nan)

In [ ]:
# Extract category information

yelp_clean["categories"] = yelp_df["categories"].apply(lambda x: ", ".join([d["title"] for d in x]) if isinstance(x, list) else np.nan)
yelp_clean["main_category"] = yelp_df["categories"].apply(lambda x: x[0]["title"] if isinstance(x, list) and len(x) > 0 else np.nan)

In [ ]:
# Extract transaction information

yelp_clean["transactions"] = yelp_df["transactions"].apply(lambda x: ", ".join(x) if isinstance(x, list) else np.nan)

In [ ]:
# Build service dummy variables

yelp_clean["has_delivery"] = yelp_clean["transactions"].apply(lambda x: 1 if isinstance(x, str) and "delivery" in x else 0)
yelp_clean["has_pickup"] = yelp_clean["transactions"].apply(lambda x: 1 if isinstance(x, str) and "pickup" in x else 0)
yelp_clean["has_reservation"] = yelp_clean["transactions"].apply(lambda x: 1 if isinstance(x, str) and "restaurant_reservation" in x else 0)

**convert price sign to numeric level**

In [ ]:
# Convert price sign to numeric level
# $ = 1, $$ = 2, $$$ = 3, $$$$ = 4

def price_to_num(x):
  if pd.isna(x):
    return np.nan
  return len(str(x))

yelp_clean["price_level"] = yelp_clean["price"].apply(price_to_num)

**keep Manhattan Zip**

In [ ]:
# Make sure zip_code and main_category exist in yelp_clean

print("Current yelp_clean columns BEFORE fix:")
print(list(yelp_clean.columns))

# If zip_code does not exist, create it from yelp_df["location"]
if "zip_code" not in yelp_clean.columns:
  yelp_clean["zip_code"] = yelp_df["location"].apply(lambda x: x.get("zip_code") if isinstance(x, dict) else np.nan)

# If main_category does not exist, create it from yelp_df["categories"]
if "main_category" not in yelp_clean.columns:
  yelp_clean["main_category"] = yelp_df["categories"].apply(lambda x: x[0]["title"] if isinstance(x, list) and len(x) > 0 else np.nan)

print("Current yelp_clean columns AFTER fix:")
print(list(yelp_clean.columns))

print(yelp_clean[["zip_code", "main_category"]].head())

In [ ]:
# Keep important variables for Yelp analysis

yelp_analysis = yelp_clean.dropna(subset=["rating", "review_count", "zip_code", "main_category"]).copy()

manhattan_zips = [
    "10001", "10002", "10003", "10004", "10005", "10006", "10007", "10009",
    "10010", "10011", "10012", "10013", "10014", "10016", "10017", "10018",
    "10019", "10020", "10021", "10022", "10023", "10024", "10025", "10026",
    "10027", "10028", "10029", "10030", "10031", "10032", "10033", "10034",
    "10035", "10036", "10037", "10038", "10039", "10040", "10280", "10282"]

yelp_analysis["zip_code"] = yelp_analysis["zip_code"].astype(str).str.strip()

yelp_analysis = yelp_analysis[yelp_analysis["zip_code"].isin(manhattan_zips)].copy()

print(yelp_analysis.shape)
print(yelp_analysis["zip_code"].value_counts().head(20))

**handle missing values**

In [ ]:
# We handle missing values differently depending on the importance of each feature.

# 1.For key variables (rating, review_count, zip_code, category), we drop rows with missing values.
# 2.For optional variables (price, services), we keep missing values or handle them separately.
# 3.For inspection data, we only drop missing values when analyzing specific variables (e.g., score).

In [ ]:
# Yelp missing values
# Not all restaurants show to pulic their price level, which means this variable have some natural sample bias

yelp_clean.isna().sum()

In [ ]:
# instead of applying dropna, we decide to keep these rows when doing other analysis
# we will use price_df that drop those empty values only when we need to deal with 'price level' in our code

price_df = yelp_analysis.dropna(subset=["price_level"]).copy()

In [ ]:
# Yelp cleaned info

yelp_clean.info()

In [ ]:
# Show Yelp cleaned columns

print("Yelp clean columns:")
print(list(yelp_clean.columns))

In [ ]:
yelp_analysis.head()

### **NYC Open Data Cleaning**

In [ ]:
# NYC Open Data cleaning

inspection_clean["zipcode"] = inspection_clean["zipcode"].astype(str).str.replace(".0", "", regex=False)
inspection_clean["zipcode"] = inspection_clean["zipcode"].str.strip()

inspection_clean.info()

**handle missing values**

In [ ]:
# NYC missing values

inspection_clean.isna().sum()

In [ ]:
# Drop clearly unusable rows in NYC data

inspection_clean = inspection_clean[inspection_clean["zipcode"] != "nan"].copy()
inspection_clean = inspection_clean.dropna(subset=["cuisine_description"]).copy()

print(inspection_clean.shape)
inspection_clean.head()

**convert inspection row-level data to restaurant-level latest inspection**

In [ ]:
# Build restaurant-level NYC inspection table
# Important:
# Raw inspection rows are not restaurant-level rows.
# One inspection can have multiple violation rows.
# So we first aggregate each inspection event, then keep the latest
# inspection for each restaurant.

inspection_clean["is_critical"] = inspection_clean["critical_flag"].apply(lambda x: 1 if x == "Critical" else 0)

inspection_event = inspection_clean.groupby(["camis", "dba", "zipcode", "cuisine_description", "inspection_date", "grade", "score"]).agg({"violation_code": "count","is_critical": "sum"}).reset_index()

inspection_event = inspection_event.rename(columns={"violation_code": "violation_count","is_critical": "critical_violation_count"})

inspection_event = inspection_event.sort_values("inspection_date")

inspection_latest = inspection_event.drop_duplicates(subset="camis", keep="last").copy()

print("inspection_event shape:", inspection_event.shape)
print("inspection_latest shape:", inspection_latest.shape)
inspection_latest.head()

In [ ]:
# Remove missing values for the latest-inspection table

inspection_latest = inspection_latest.dropna(subset=["zipcode", "cuisine_description", "inspection_date", "score"]).copy()

print(inspection_latest.shape)
inspection_latest.head()

In [ ]:
inspection_latest["year"] = inspection_latest["inspection_date"].dt.year
inspection_latest["month"] = inspection_latest["inspection_date"].dt.month

## **Step 3 analytical lens definition**

In [ ]:
# Consumer appeal metrics: average rating; review_count; high-rating subset (rating >= 4.5)

In [ ]:
# Operational stability metrics: inspection score（lower is better）; grade distribution; critical violation

## **Step 4 Core Analysis**

### **I — Yelp: overall market landscape**

**rating distribution**

In [ ]:
# Most restaurants are concentrated between 4.0 and 4.5 ratings, indicating strong competition in Manhattan.

plt.figure(figsize=(8, 5))
sns.histplot(yelp_analysis["rating"], bins=10)
plt.title("Distribution of Yelp Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()

**review count distribution**

In [ ]:
# Review counts are highly skewed, with a small number of very popular restaurants.

plt.figure(figsize=(8, 5))
sns.histplot(np.log1p(yelp_analysis["review_count"]), bins=30)
plt.title("Log Distribution of Review Counts")
plt.xlabel("Log(1 + Review Count)")
plt.ylabel("Count")
plt.show()

### **II — Cuisine is the main segmentation variable**

**cuisine vs average rating**

In [ ]:
# We only keep cuisines with enough observations

cuisine_count = yelp_analysis["main_category"].value_counts()
top_cuisines = cuisine_count[cuisine_count >= 15].index

cuisine_df = yelp_analysis[yelp_analysis["main_category"].isin(top_cuisines)].copy()

cuisine_summary = cuisine_df.groupby("main_category").agg({
  "rating": "mean",
  "review_count": "mean",
  "price_level": "mean",
  "id": "count"}).rename(columns={"id": "restaurant_count"})

cuisine_summary = cuisine_summary.sort_values("rating", ascending=False)

print(cuisine_summary.head(20))

In [ ]:
# Seafood、Sushi Bars、Ramen、Japanese、Korean、Italian 是 Yelp 上表现最强的一批 cuisine。
# Italian worth to be emphasized the most, with 4.26 rating under 139 samples, indicating strong reliability and consistency
# Sushi Bars、Korean、Thai have a high propotion of 4.5+ ratings, indicating that restaurants of high ratings have higher probability to appear within these cuisine types
# The advantages of Italian、Ramen are not just high rating avergae, but also relative solid in consistency
# Mediterranean have high averge but the sample is too small(16), can only be seen as “promising but less conclusive”。

plt.figure(figsize=(12, 6))
sns.barplot(
    data=cuisine_summary.reset_index(),
    x="main_category",
    y="rating"
)
plt.title("Average Rating by Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("Average Rating")
plt.xticks(rotation=45)
plt.show()

**cuisine rating distribution**

In [ ]:
# Only focusing on the average may be misleading, if two cisine types have same average but one have thinner box in the plot and the other have fatter one
# More consistent and predicatable ones are those that have high average, thin box, and few outliers

plt.figure(figsize=(12, 6))
sns.boxplot(data=cuisine_df, x="main_category", y="rating")
plt.title("Rating Distribution by Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("Rating")
plt.xticks(rotation=45)
plt.show()

**cuisine rating structure**

In [ ]:
# Is the high rating because of most restaurants have high ratings, or due to only a few restaurants have extremely high ratings？
# Is a high average rating caused by widespread strong ratings, or by only a few very highly rated restaurants?

rating_structure = cuisine_df.groupby("main_category").agg(
    mean_rating=("rating", "mean"),
    std_rating=("rating", "std"),
    restaurant_count=("id", "count"),
    share_4plus=("rating", lambda x: (x >= 4.0).mean() * 100),
    share_45plus=("rating", lambda x: (x >= 4.5).mean() * 100),
    share_5only=("rating", lambda x: (x == 5.0).mean() * 100))

rating_structure = rating_structure.sort_values("mean_rating", ascending=False)

print(rating_structure.head(20))

In [ ]:
# Focus on top cuisines by average rating

focus_cuisines = list(rating_structure.head(8).index)

rating_band_df = cuisine_df[cuisine_df["main_category"].isin(focus_cuisines)].copy()

rating_band_table = rating_band_df.groupby(["main_category", "rating"])["id"].count().unstack(fill_value=0)

rating_band_pct = rating_band_table.div(rating_band_table.sum(axis=1), axis=0) * 100

print(rating_band_pct)

In [ ]:
# Stacked bar chart: rating composition

rating_band_pct.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Rating Composition by Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("Percent of Restaurants")
plt.xticks(rotation=45)
plt.legend(title="Rating", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

**cuisine vs review_count**

In [ ]:
# Ramen review_count is high，indicating high popularity
# French have high review，but the rating isn't necessary the highest
# Italian have both high rating and high review_count，is worth focusing.

cuisine_review_summary = cuisine_df.groupby("main_category").agg({
    "review_count": "mean",
    "rating": "mean",
    "id": "count"
}).rename(columns={"id": "restaurant_count"})

cuisine_review_summary = cuisine_review_summary.sort_values("review_count", ascending=False)

print(cuisine_review_summary.head(20))

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=cuisine_review_summary.reset_index(),
    x="main_category",
    y="review_count")
plt.title("Average Review Count by Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("Average Review Count")
plt.xticks(rotation=45)
plt.show()

### **III — Location (Zip code)**

In [ ]:
# First look at restaurant count
# Restaurants in Manhattan are concentrated highly in several business centres.

zip_counts = yelp_analysis["zip_code"].value_counts().head(10)

print(zip_counts)

zip_counts.plot(kind="bar", figsize=(10, 6))
plt.title("Restaurant Count by ZIP Code")
plt.xlabel("ZIP Code")
plt.ylabel("Count")
plt.show()

In [ ]:
# ZIP code summary:
# For investment analysis, count alone is not enough.
# We need average rating and average review_count too.

zip_summary = yelp_analysis.groupby("zip_code").agg({
    "rating": "mean",
    "review_count": "mean",
    "id": "count"
}).rename(columns={"id": "restaurant_count"})

zip_summary = zip_summary[zip_summary["restaurant_count"] >= 10].copy()
zip_summary = zip_summary.sort_values(["rating", "restaurant_count"], ascending=[False, False])

print(zip_summary.head(15))

In [ ]:
# Top 5 ZIP codes: deeper cuisine analysis
# Goal: find possible cuisine gaps and neighborhood specialization

top5_zip = list(zip_summary.head(5).index)

zip_cuisine = yelp_analysis[
    yelp_analysis["zip_code"].isin(top5_zip)
].groupby(["zip_code", "main_category"]).agg({
    "rating": "mean",
    "review_count": "mean",
    "id": "count"}).rename(columns={"id": "restaurant_count"}).reset_index()

zip_cuisine = zip_cuisine[zip_cuisine["restaurant_count"] >= 2].copy()

print(zip_cuisine.sort_values(["zip_code", "rating"], ascending=[True, False]).head(50))

In [ ]:
# ZIP code with most restaurants include 10019、10003、10016、10036、10022、10001。
# but is we look at average rating，10029、10002、10009、10007、10014 are more advantaged。
# Thus generally, zip codes with both high average ratings and restaurant numbers are 10019、10003、10016，followed by 10002、10018、10014。

In [ ]:
# Zip code affects market performance but more indicating partial market enviroment instead of individual quality

### **IV — Supporting factors: price and review counts**

**price**

In [ ]:
# Higher price levels tend to have slightly higher ratings, but the difference is not dramatic.

price_df = yelp_analysis.dropna(subset=["price_level"]).copy()

price_summary = price_df.groupby("price_level").agg({
    "rating": "mean",
    "review_count": "mean",
    "id": "count"}).rename(columns={"id": "restaurant_count"})

print(price_summary)

plt.figure(figsize=(8, 5))
sns.boxplot(data=price_df, x="price_level", y="rating")
plt.title("Rating by Price Level")
plt.xlabel("Price Level")
plt.ylabel("Rating")
plt.show()

In [ ]:
# Fine dining are more likely to receive high ratings高端餐厅更容易获得更高评分。
# but we cannot conclude higher price-higher rating as the difference in 1.0 to 3.0 is unlinear
# Furthermore, there are many empty values, so we only see this as a subsample analysis

**review** **count**

In [ ]:
# No strong relationship

plt.figure(figsize=(8, 5))
sns.regplot(data=yelp_analysis, x="review_count", y="rating", scatter_kws={"alpha": 0.4})
plt.title("Review Count vs Rating")
plt.xlabel("Review Count")
plt.ylabel("Rating")
plt.show()

**Delivery / Pickup / Reservation vs Rating**

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=yelp_analysis, x="has_delivery", y="rating")
plt.title("Delivery vs Rating")
plt.xlabel("Has Delivery")
plt.ylabel("Rating")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=yelp_analysis, x="has_pickup", y="rating")
plt.title("Pickup vs Rating")
plt.xlabel("Has Pickup")
plt.ylabel("Rating")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=yelp_analysis, x="has_reservation", y="rating")
plt.title("Reservation vs Rating")
plt.xlabel("Has Reservation")
plt.ylabel("Rating")
plt.show()

### **V — Operational stability from NYC inspection data**

In [ ]:
# Important: lower score is better

**Inspection score distribution**

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(inspection_latest["score"], bins=30)
plt.title("Inspection Score Distribution")
plt.xlabel("Inspection Score")
plt.ylabel("Count")
plt.show()

**Grade distribution**

In [ ]:
grade_summary = inspection_latest["grade"].value_counts(dropna=False)
print(grade_summary)

plt.figure(figsize=(8, 5))
sns.countplot(data=inspection_latest, x="grade", order=inspection_latest["grade"].value_counts().index)
plt.title("Grade Distribution")
plt.xlabel("Grade")
plt.ylabel("Count")
plt.show()

**Cuisine vs inspection score**

In [ ]:
# Keep cuisines with enough observations
# some cuisine market are more likely to keep low inspection score
# Cuisine types with more standardized operations and lower preparation complexity tend to manage hygiene risks more effectively such as beverages, desserts, and standardized fast-casual formats.
# In contrast, cuisines with higher preparation complexity, longer ingredient chains, and stricter cold and hot storage requirements face greater challenges in maintaining low inspection scores.

top_ins_cuisines = inspection_latest["cuisine_description"].value_counts()
top_ins_cuisines = top_ins_cuisines[top_ins_cuisines >= 15].index

ins_cuisine_df = inspection_latest[
    inspection_latest["cuisine_description"].isin(top_ins_cuisines)
].copy()

ins_cuisine_summary = ins_cuisine_df.groupby("cuisine_description").agg({
    "score": "mean",
    "violation_count": "mean",
    "critical_violation_count": "mean",
    "camis": "count"
}).rename(columns={"camis": "restaurant_count"})

ins_cuisine_summary = ins_cuisine_summary.sort_values("score")

print(ins_cuisine_summary.head(20))

In [ ]:
plt.figure(figsize=(12, 14))

plot_data = ins_cuisine_summary.reset_index().sort_values("score", ascending=True)

sns.barplot(data=plot_data, y="cuisine_description", x="score")

plt.title("Average Inspection Score by Cuisine (Lower is Better)")
plt.xlabel("Average Inspection Score")
plt.ylabel("Cuisine")
plt.tight_layout()
plt.show()

In [ ]:
cuisine_count = ins_cuisine_df["cuisine_description"].value_counts()
top10_cuisine = cuisine_count.head(10).index

plot_data = ins_cuisine_df[ins_cuisine_df["cuisine_description"].isin(top10_cuisine)]

plt.figure(figsize=(12, 10))
sns.boxplot(data=plot_data, y="cuisine_description", x="score")

plt.title("Inspection Score Distribution by Top 10 Cuisines")
plt.xlabel("Inspection Score")
plt.ylabel("Cuisine")
plt.tight_layout()
plt.show()

**critical violations vs score**

In [ ]:
# Critical violations vs score

plt.figure(figsize=(8, 5))
sns.regplot(data=inspection_latest, x="critical_violation_count", y="score", scatter_kws={"alpha": 0.4})
plt.title("Critical Violations vs Inspection Score")
plt.xlabel("Critical Violation Count")
plt.ylabel("Inspection Score")
plt.show()

**most common violation types**

In [ ]:
# We use raw inspection_clean here because violation_description is a row-level variable

top_violations = inspection_clean["violation_description"].value_counts().head(10)

print(top_violations)

In [ ]:
# We can mainly conclude violation risk into 4 types:
# sanitation and equipment maintenance, pest control, temperature control, and cross-contamination and food safety procedures.

plt.figure(figsize=(10, 6))
top_violations.sort_values().plot(kind="barh")
plt.title("Top 10 Most Common Violation Types in Manhattan")
plt.xlabel("Count")
plt.ylabel("Violation Description")
plt.show()

## **Step 5 Synthesis — Which restaurant segments look most promising?**

A. safer / more stable segments

(Not necessary best from Yelp dataset only
but inspection score is more solid
better price
their business mode are easier to copy)

Coffee/Tea

Cafes

Bakeries

some healthy food/drinks

B. higher-upside but more execution-sensitive

(strong in Yelp
high attraction to consumers
may be more complex to operate，and not necessarily best in inspection)


Italian

Sushi Bars / Japanese

Korean

Ramen

Thai

C. Location conclusion


10019 / 10003 / 10016：markets that are more mature with larger scale

10002 / 10014 / 10029：area with opprtunities to appear high rating and specialty

## **Step 6 Additional research questions for depth**

### **Q1. Correlation analysis**

In [ ]:
# Correlation matrix for rating-related variables
# which variable is more correlated with rating linearly
# price_level and rating have some positive correlation
# review_count and rating may be weaker positive correlation
# distance and rating have weak correlation
# service-related variables and rating almost don't have linear correkation

# note that correlation doesn't mean causation

yelp_analysis["log_review_count"] = np.log1p(yelp_analysis["review_count"])

corr_df = yelp_analysis[
    ["rating", "review_count", "log_review_count", "distance",
     "price_level", "has_delivery", "has_pickup", "has_reservation"]
].dropna()

corr_matrix = corr_df.corr()

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Correlation Matrix for Yelp Success Variables")
plt.show()

### **Q2. Zip code with success**

In [ ]:
# ZIP code x cuisine rating heatmap

zip_cuisine_pivot = zip_cuisine.pivot(
    index="main_category",
    columns="zip_code",
    values="rating"
)

plt.figure(figsize=(10, 8))
sns.heatmap(zip_cuisine_pivot, annot=True, cmap="YlGnBu")
plt.title("Cuisine Rating Heatmap Across Top 5 ZIP Codes")
plt.xlabel("ZIP Code")
plt.ylabel("Cuisine")
plt.show()

In [ ]:
# ZIP code x cuisine restaurant count heatmap
# This helps detect possible local cuisine gaps

zip_cuisine_count_pivot = zip_cuisine.pivot(
    index="main_category",
    columns="zip_code",
    values="restaurant_count"
)

plt.figure(figsize=(10, 8))
sns.heatmap(zip_cuisine_count_pivot, annot=True, cmap="Blues")
plt.title("Cuisine Restaurant Count Heatmap Across Top 5 ZIP Codes")
plt.xlabel("ZIP Code")
plt.ylabel("Cuisine")
plt.show()

### **Q3. Within the same price level, which cuisines perform better?**

In [ ]:
cuisine_price_summary = price_df.groupby(["price_level", "main_category"]).agg(
    mean_rating=("rating", "mean"),
    mean_review_count=("review_count", "mean"),
    restaurant_count=("id", "count")
).reset_index()

# keep only groups with enough observations
cuisine_price_summary = cuisine_price_summary[
    cuisine_price_summary["restaurant_count"] >= 5
].copy()

print(cuisine_price_summary.sort_values(["price_level", "mean_rating"], ascending=[True, False]).head(40))

In [ ]:
# Show top cuisines within each price level

for p in sorted(cuisine_price_summary["price_level"].dropna().unique()):
    print("\n" + "=" * 60)
    print("Price level:", p)
    print("=" * 60)

    one_price = cuisine_price_summary[cuisine_price_summary["price_level"] == p].copy()
    one_price = one_price.sort_values("mean_rating", ascending=False)

    print(one_price.head(10))

In [ ]:
# Visualization: top 5 cuisines within each price level

for p in sorted(cuisine_price_summary["price_level"].dropna().unique()):
    one_price = cuisine_price_summary[cuisine_price_summary["price_level"] == p].copy()
    one_price = one_price.sort_values("mean_rating", ascending=False).head(5)

    plt.figure(figsize=(8, 5))
    sns.barplot(data=one_price, x="main_category", y="mean_rating")
    plt.title("Top 5 Cuisines within Price Level " + str(int(p)))
    plt.xlabel("Cuisine")
    plt.ylabel("Average Rating")
    plt.xticks(rotation=45)
    plt.show()

### **Q4. Common dishes / ingredient characteristics in highly rated cuisines**

In [ ]:
# Common dish patterns in highly rated cuisines

# IMPORTANT:
# This is a cuisine-level supplementary analysis.
# It does NOT claim to represent the exact menus of Manhattan restaurants.
# Instead, it uses dish titles from a supplementary API to identify
# common dish keywords and dish phrases in highly rated cuisines.

In [ ]:
# Match Yelp high-rating cuisines to cuisines available in TheMealDB

available_area_map = {
    "Thai": "Thai",
    "Italian": "Italian",
    "Indian": "Indian",
    "Japanese": "Japanese",
    "Mexican": "Mexican",
    "American": "American"}

In [ ]:
# Build a cuisine table from Yelp results
# We only keep cuisines with enough observations

top_cuisine_table = cuisine_summary.reset_index().copy()
top_cuisine_table = top_cuisine_table[top_cuisine_table["restaurant_count"] >= 15].copy()
top_cuisine_table = top_cuisine_table.sort_values("rating", ascending=False)

print("Top Yelp cuisines with enough observations:")
print(top_cuisine_table.head(20))

In [ ]:
# Keep only cuisines that can be matched to TheMealDB

matched_cuisines = []

for cuisine_name in list(top_cuisine_table["main_category"]):
  if cuisine_name in available_area_map:
    matched_cuisines.append(cuisine_name)

print("Matched cuisines for Step 8:")
print(matched_cuisines)

In [ ]:
# Select top matched cuisines for analysis

selected_planA_cuisines = matched_cuisines[:6]

print("Selected cuisines for dish-pattern analysis:")
print(selected_planA_cuisines)

In [ ]:
# Function to get meals by cuisine area from TheMealDB

def get_area_meals(area_name):
    url = "https://www.themealdb.com/api/json/v1/1/filter.php?a=" + area_name
    response = requests.get(url)

    if response.status_code == 200:
      data = response.json()

      if "meals" in data and data["meals"] is not None:
        return data["meals"]
      else:
        return []
    else:
      return []

In [ ]:
# Stopwords: words we do NOT want to dominate the result
# We remove seasoning / generic words / less meaningful words

dish_stopwords = [
    "and", "with", "the", "a", "an", "of", "in", "on", "style",
    "garlic", "salt", "pepper", "oil", "olive", "butter", "sauce",
    "red", "green", "black", "white", "fresh", "hot", "cold",
    "classic", "traditional", "easy", "best"]

In [ ]:
# First, show the raw dish titles in each cuisine

for cuisine_name in selected_planA_cuisines:
    print("\n" + "=" * 70)
    print("Cuisine:", cuisine_name)
    print("=" * 70)

    area_name = available_area_map[cuisine_name]
    meals = get_area_meals(area_name)

    print("Number of meals returned by API:", len(meals))

    meal_title_list = []

    for meal in meals:
        meal_title_list.append(meal["strMeal"])

    meal_title_df = pd.DataFrame({"dish_title": meal_title_list})

    print("\nSample dish titles:")
    print(meal_title_df.head(20))

In [ ]:
# Count dish keywords from meal titles
# break titles into words and remove generic seasoning words

for cuisine_name in selected_planA_cuisines:
    print("\n" + "=" * 70)
    print("Cuisine:", cuisine_name)
    print("=" * 70)

    area_name = available_area_map[cuisine_name]
    meals = get_area_meals(area_name)

    word_count = {}

    for meal in meals:
        title = meal["strMeal"].lower()

        title = title.replace("-", " ")
        title = title.replace("&", " ")
        title = title.replace("/", " ")
        title = title.replace("(", " ")
        title = title.replace(")", " ")
        title = title.replace(",", " ")
        title = title.replace(".", " ")

        words = title.split()

        for w in words:
            if len(w) > 2 and w not in dish_stopwords:
                if w in word_count:
                    word_count[w] = word_count[w] + 1
                else:
                    word_count[w] = 1

    word_df = pd.DataFrame({
        "dish_keyword": list(word_count.keys()),
        "count": list(word_count.values())
    })

    word_df = word_df.sort_values("count", ascending=False)

    print("\nTop dish keywords:")
    print(word_df.head(15))

    plt.figure(figsize=(10, 6))
    sns.barplot(data=word_df.head(15), x="count", y="dish_keyword")
    plt.title("Top Dish Keywords in " + cuisine_name + " Cuisine")
    plt.xlabel("Count")
    plt.ylabel("Dish Keyword")
    plt.show()

In [ ]:
# Count dish phrases (bigrams)
# This is more like actual dish-type signals than single words


for cuisine_name in selected_planA_cuisines:
    print("\n" + "=" * 70)
    print("Cuisine:", cuisine_name)
    print("=" * 70)

    area_name = available_area_map[cuisine_name]
    meals = get_area_meals(area_name)

    bigram_count = {}

    for meal in meals:
        title = meal["strMeal"].lower()

        title = title.replace("-", " ")
        title = title.replace("&", " ")
        title = title.replace("/", " ")
        title = title.replace("(", " ")
        title = title.replace(")", " ")
        title = title.replace(",", " ")
        title = title.replace(".", " ")

        words = title.split()

        clean_words = []
        for w in words:
            if len(w) > 2 and w not in dish_stopwords:
                clean_words.append(w)

        for i in range(len(clean_words) - 1):
            bg = clean_words[i] + " " + clean_words[i + 1]

            if bg in bigram_count:
                bigram_count[bg] = bigram_count[bg] + 1
            else:
                bigram_count[bg] = 1

    bigram_df = pd.DataFrame({
        "dish_phrase": list(bigram_count.keys()),
        "count": list(bigram_count.values())
    })

    bigram_df = bigram_df.sort_values("count", ascending=False)

    print("\nTop dish phrases:")
    print(bigram_df.head(15))

    plt.figure(figsize=(10, 6))
    sns.barplot(data=bigram_df.head(15), x="count", y="dish_phrase")
    plt.title("Top Dish Phrases in " + cuisine_name + " Cuisine")
    plt.xlabel("Count")
    plt.ylabel("Dish Phrase")
    plt.show()